# Ordered Logistic Regression Results Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR²](https://sen.science/doi/10.71728/senscience.y7m0-f273) dataset using the `mlcroissant` library.

### Dataset Source
Dataset Croissant metadata URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

We'll examine record sets, fields, process data, and perform exploratory analyses.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant metadata schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'
# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Let's review available record sets (`@id`), their human-friendly names, and their fields. We will use only `@id` references for all programmatic access.

In [ ]:
# List available record sets and their fields (by @id)
record_sets = dataset.record_sets
if len(record_sets) == 0:
    print("No record sets found in this dataset's schema.")
else:
    for rs in record_sets:
        print(f"\nRecord set:   @id = '{rs['@id']}'")
        print(f"              name = {rs.get('name', '-')}")
        print("  Fields:")
        for field in rs.get('field', []):
            print(f"    @id = '{field['@id']}', name = {field.get('name','-')}, dataType = {field.get('dataType','-')}")

## 3. Data Extraction
Load data from each record set into a pandas DataFrame. Reference record sets and fields by their `@id`s, as above.

In [ ]:
# Extract all record sets to DataFrames.
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records from record set '@id': {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"  Loaded {len(dataframes[record_set_id])} records. Columns (@id): {list(dataframes[record_set_id].columns)}")
        else:
            print("  No records available.")
    except Exception as e:
        print(f"  Could not load: {e}")
# Display the first few rows of the first available record set
if len(dataframes) > 0:
    first_rs = next(iter(dataframes.keys()))
    print(f"\nFirst record set loaded: {first_rs} - showing first 5 records:")
    display(dataframes[first_rs].head())
else:
    print("No data could be loaded into dataframes.")

## 4. Exploratory Data Analysis (EDA)
Let's select a numeric field in one of the record sets (referenced by `@id`) for basic analysis: filtering, normalization, and grouping.

**Instructions:**
- Pick a record set and field `@id` from the overview above for the examples below, replacing placeholder strings if necessary.
- If no numeric field is available, adapt with an appropriate field for demo purposes.

In [ ]:
# Example: Perform EDA on a numeric field (update the '@id's as needed based on your schema!)
if len(dataframes) > 0:
    record_set_id = next(iter(dataframes.keys()))
    df = dataframes[record_set_id]
    # Pick a numeric field. (List numeric candidates: float or int columns.)
    numeric_candidates = df.select_dtypes(include=['number']).columns
    if len(numeric_candidates) == 0:
        print("No numeric field found in this record set; showing column names:")
        print(list(df.columns))
    else:
        numeric_field_id = numeric_candidates[0]
        print(f"Using numeric field '@id': {numeric_field_id}")
        threshold = df[numeric_field_id].quantile(0.75)  # use 75th percentile as threshold for demo
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records where '{numeric_field_id}' > {threshold}:")
        display(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Try group by a categorical/textual field
        group_field_candidates = df.select_dtypes(include=['object']).columns
        if len(group_field_candidates) > 0:
            group_field_id = group_field_candidates[0]
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean of '{numeric_field_id}' by '{group_field_id}':")
            display(grouped_df.head())
        else:
            print("No categorical/text column found to group by.")
else:
    print("No dataframes available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields (using `matplotlib`/`seaborn`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if len(dataframes) > 0:
    df = next(iter(dataframes.values()))
    numeric_cols = df.select_dtypes(include=['number']).columns
    if len(numeric_cols) > 0:
        plt.figure(figsize=(7, 4))
        sns.histplot(df[numeric_cols[0]], kde=True)
        plt.title(f"Distribution of '{numeric_cols[0]}'")
        plt.xlabel(numeric_cols[0])
        plt.show()

        if len(numeric_cols) > 1:
            plt.figure(figsize=(7, 5))
            sns.scatterplot(data=df, x=numeric_cols[0], y=numeric_cols[1])
            plt.title(f"Scatter of '{numeric_cols[0]}' vs '{numeric_cols[1]}'")
            plt.xlabel(numeric_cols[0])
            plt.ylabel(numeric_cols[1])
            plt.show()
    else:
        print("No numeric columns found for visualization.")
else:
    print("No dataframes to visualize.")

## 6. Conclusion

- Loaded and explored the [FAIR²](https://sen.science/doi/10.71728/senscience.y7m0-f273) rangeland management dataset using `mlcroissant` and Croissant schema.
- Reviewed record set and field structure with all programmatic references by `@id`.
- Extracted records, demonstrated filtering and normalization for numeric analysis.
- Visualized sample data distribution.

This notebook serves as a blueprint for analyzing multi-recordset Croissant datasets with strongly referenced schema identifiers.